In [ ]:
# Cell 1 — #014 setup + data: FTMO macro universe daily close panel.
# Source: data/ftmo_daily.parquet (Dukascopy daily BID closes 2000-2025 + 2026 H1 from
# hourly; 45 instruments: 28 FX, 12 indices, 2 metals, 3 energy; crypto EXCLUDED — swap
# -30%/yr per side). Instruments start at different dates (GER40/US30 2013, NATGAS 2012,
# UKOIL 2006) -> ragged head is expected; ffill capped at 5 days so dead/stale series
# don't ghost-trade; leading NaNs stay NaN until an instrument is born.
import numpy as np
import pandas as pd
from backtest.data import DataPanel

RAW = pd.read_parquet(r"C:\Users\User\backtest_engine\backtest_engine2\data\ftmo_daily.parquet")
RAW["date"] = pd.to_datetime(RAW["date"]).dt.tz_localize(None)

px = RAW.pivot_table(index="date", columns="symbol", values="close", aggfunc="last")
px = px[sorted(px.columns)]
# Sunday rows: FX/crypto venues print a thin Sunday candle; indices don't. Collapse to a
# weekday grid (Mon-Fri), Sunday's close is superseded by Monday anyway at daily horizon.
px = px[px.index.dayofweek < 5]
grid = pd.bdate_range(px.index[0], px.index[-1])
prices = px.reindex(grid).ffill(limit=5)

panel = DataPanel(prices, check_outliers=False)  # known real jumps: CHF depeg 2015, COVID, oil 2020

print(f"Panel: {len(panel.dates)} days x {len(panel.assets_all)} instruments | "
      f"{panel.dates[0].date()} -> {panel.dates[-1].date()}")
born = prices.notna().idxmax()
late = born[born > prices.index[0] + pd.Timedelta(days=30)]
print(f"Late starters: {dict((k, str(v.date())) for k, v in late.items())}")
alive = prices.notna().sum(axis=1)
print(f"Instruments alive: start {alive.iloc[0]}, 2010 {alive[alive.index.year == 2010].iloc[0]}, "
      f"end {alive.iloc[-1]}")
print(f"Last dates with data per instrument: min {prices.apply(lambda c: c.last_valid_index()).min().date()}")

# Sanity: known levels at known dates (rough).
checks = [("EURUSD", "2008-07-15", 1.55, 1.62), ("US500", "2020-03-23", 2100, 2400),
          ("XAUUSD", "2011-09-05", 1850, 1950), ("USOIL", "2008-07-03", 135, 150),
          ("JP225", "2024-07-11", 41000, 43000)]
for sym, d, lo, hi in checks:
    v = prices.loc[d, sym]
    print(f"  {sym} @ {d}: {v:,.2f} {'OK' if lo <= v <= hi else f'!! expected [{lo},{hi}]'}")
prices.tail(2).iloc[:, :6]


In [ ]:
# Cell 2 — #014 trial 1 strategy: MOP-exact TSMOM (pre-registered).
# sign(252d return) per instrument; w_i = sign_i * (0.40 / sigma_i) / N_alive;
# sigma_i = 60d EWMA ann vol, FLOORED at 5% (peg landmine: EURCHF 2012-14 vol ~1% ->
# 8x position -> Jan-2015 depeg -17%/day; floor is a pre-registered risk rule, not tuning).
# Monthly rebalance (last business day); <252d history -> sit out.
from backtest.strategy import Strategy
from backtest.risk import RiskConfig

LOOKBACK = 252
VOL_SPAN = 60
VOL_FLOOR = 0.05
SCALE = 0.40


def _month_end_bars(dates):
    s = pd.Series(dates, index=dates)
    return pd.DatetimeIndex(s.groupby([dates.year, dates.month]).last().values)


class TSMOM(Strategy):
    """MOP 2012: sign(12m return), 40%/sigma sizing, equal split across live signals."""
    rebalance_frequency = staticmethod(_month_end_bars)

    def __init__(self):
        self.risk = RiskConfig(max_position=0.25, max_gross=6.0)

    def __repr__(self):
        return "TSMOM(sign 252d, 0.40/sigma, vol floor 5%, monthly)"

    def generate_weights(self, data, t):
        px = data.prices
        if len(px) < LOOKBACK + 1:
            return pd.Series(0.0, index=data.assets)
        sig = np.sign(px.iloc[-1] / px.iloc[-(LOOKBACK + 1)] - 1.0)
        vol = (px.pct_change(fill_method=None).ewm(span=VOL_SPAN).std().iloc[-1]
               * np.sqrt(252)).clip(lower=VOL_FLOOR)
        w = (sig * SCALE / vol).fillna(0.0)
        n_alive = int((w != 0).sum())
        if n_alive == 0:
            return pd.Series(0.0, index=data.assets)
        return (w / n_alive).reindex(data.assets).fillna(0.0)


strat = TSMOM()
print(strat)
rb = strat.rebalance_dates(panel.dates)
print(f"Rebalance dates: {len(rb)} month-ends, {rb[0].date()} -> {rb[-1].date()}")

# Dry-run the weight function at three sample dates for sanity.
from backtest.data import DataView
for d in [rb[13], rb[150], rb[-1]]:
    w = strat.generate_weights(DataView(panel, d), d)
    live = w[w != 0]
    print(f"{d.date()}: {len(live)} live | gross {live.abs().sum():.2f} | "
          f"long {int((live > 0).sum())} short {int((live < 0).sum())} | "
          f"max |w| {live.abs().max():.3f} ({live.abs().idxmax()})")


In [ ]:
# Cell 3 — #014 real FTMO cost stack from the measured spec table (no sweeps).
# Spread: per-asset half-spread from demo quotes, FLOORED at 0.25bp half (demo prints ~0
# on some indices). Swap: custom holding-cost model, per-instrument AND per-direction,
# annual rate / 252 per daily bar; rates already include the 3x-Wednesday convention;
# credits (USDJPY long, US30 short) are real and count for us. No commission (FX/index/
# metals commission-free), no impact/liquidity cap (size trivial vs FX/index depth).
# Missing instruments get their ASSET-CLASS MEDIAN rates and are flagged loudly.
from backtest.costs import Spread, CompositeCostModel
from backtest.costs.base import CostModel

SPECS = pd.read_parquet(r"C:\Users\User\backtest_engine\backtest_engine2\data\ftmo_specs.parquet")
SPECS["clean"] = SPECS["symbol"].str.replace(".cash", "", regex=False).str.upper()

CLASS = {s: ("fx" if len(s) == 6 and s[:3].isalpha() and not s.startswith(("XA", "UK", "US", "NA"))
        else "other") for s in prices.columns}
CLASS.update({s: "index" for s in ["US500", "US100", "US30", "GER40", "EU50", "FRA40",
                                   "UK100", "JP225", "AUS200", "HK50", "SPA35", "SWI20"]})
CLASS.update({"XAUUSD": "metal", "XAGUSD": "metal", "UKOIL": "energy", "USOIL": "energy",
              "NATGAS": "energy"})

half_bp, swp_l, swp_s, filled = {}, {}, {}, []
for sym in prices.columns:
    row = SPECS[SPECS["clean"] == sym]
    if row.empty:
        filled.append(sym)
        continue
    r = row.iloc[0]
    p = prices[sym].dropna().iloc[-1]
    if r["swap_mode"] == "points":
        bp_l = 1e4 * r["swap_long"] * r["point"] / p
        bp_s = 1e4 * r["swap_short"] * r["point"] / p
        swp_l[sym] = bp_l * 364 / 1e4          # annual fraction
        swp_s[sym] = bp_s * 364 / 1e4
    else:                                       # interest mode: value = annual %
        swp_l[sym] = r["swap_long"] / 100.0
        swp_s[sym] = r["swap_short"] / 100.0
    half_bp[sym] = max(1e4 * r["spread_pts"] * r["point"] / p / 2, 0.25)

for sym in filled:                               # class-median fill, flagged
    peers = [s for s in swp_l if CLASS[s] == CLASS[sym]]
    swp_l[sym] = float(np.median([swp_l[s] for s in peers]))
    swp_s[sym] = float(np.median([swp_s[s] for s in peers]))
    half_bp[sym] = float(np.median([half_bp[s] for s in peers]))
if filled:
    print(f"!! not in specs, filled with {len(filled)} class medians: {filled}")


class FTMOSwap(CostModel):
    """Per-instrument, per-direction overnight financing. Rates = annual fractions
    (negative = trader pays); charged rate/252 per daily bar on held exposure."""

    def __init__(self, long_rate: dict, short_rate: dict, trading_days: int = 252):
        self.long_rate = pd.Series(long_rate)
        self.short_rate = pd.Series(short_rate)
        self.td = trading_days

    def trade_cost(self, trades, view):
        return 0.0

    def holding_cost(self, positions, view):
        pos = positions[positions != 0]
        if pos.empty:
            return 0.0
        rate = pd.Series(np.where(pos > 0, self.long_rate.reindex(pos.index),
                                  self.short_rate.reindex(pos.index)), index=pos.index)
        return float((-rate.fillna(0.0) * pos.abs() / self.td).sum())


costs = CompositeCostModel([Spread(half_bps=half_bp), FTMOSwap(swp_l, swp_s)])
liq = None

tbl = pd.DataFrame({"class": pd.Series(CLASS), "half_bp": pd.Series(half_bp).round(2),
                    "swap_long_%yr": (pd.Series(swp_l) * 100).round(2),
                    "swap_short_%yr": (pd.Series(swp_s) * 100).round(2)}).loc[list(prices.columns)]
print(tbl.to_string())
print(f"\nbook-level: gross ~4 x avg|swap| -> rough drag ceiling "
      f"{4 * tbl[['swap_long_%yr', 'swap_short_%yr']].abs().mean().mean():.1f}%/yr if always on the expensive side")


In [ ]:
# Cell 4 — #014 trial 1: single-path run, MOP TSMOM vs real FTMO costs.
from backtest.engine import Engine

START = panel.dates[LOOKBACK + 1]
engine = Engine(panel, strat, costs=costs, liquidity=None, initial_capital=1_000_000)
result = engine.run(start=START)

eq = result.equity_curve
ret = result.returns.dropna()
yrs = (eq.index[-1] - eq.index[0]).days / 365.25
sh = ret.mean() / ret.std() * np.sqrt(252)
dd = (eq / eq.cummax() - 1).min()
print(f"Bars: {len(eq)} | rebalances {result.metadata['n_rebalances']} | "
      f"{eq.index[0].date()} -> {eq.index[-1].date()} ({yrs:.1f}y)")
print(f"Final equity:  ${eq.iloc[-1]:>14,.0f}  (total {(eq.iloc[-1] / 1e6 - 1):+.1%})")
print(f"Net Sharpe:    {sh:+.2f} | ann vol {ret.std() * np.sqrt(252):.1%} | maxDD {dd:.1%}")
print(f"Total costs:   ${result.costs.sum():>14,.0f}")
print(f"Max position:  {result.weights.abs().max().max():.1%} | "
      f"max gross {result.weights.abs().sum(axis=1).max():.2f}")

print(f"\n{'year':>5} {'net ret':>8} {'net Sh':>7} {'maxDD':>7}")
for yr, g in ret.groupby(ret.index.year):
    e = (1 + g).cumprod()
    print(f"{yr:>5} {e.iloc[-1] - 1:>+8.1%} "
          f"{g.mean() / g.std() * np.sqrt(252) if g.std() > 0 else float('nan'):>+7.2f} "
          f"{(e / e.cummax() - 1).min():>7.1%}")


In [ ]:
# Cell 5 — #014 diagnostic: gross/cost decomposition + 2022 autopsy.
# Gross = net + costs (per bar, as fraction of equity). Swap recomputed analytically
# from positions x per-side rates; spread = |trades| x half_bp (residual check vs engine).
eq = result.equity_curve
ret_net = result.returns.dropna()
cost_frac = (result.costs / eq.shift(1)).reindex(ret_net.index).fillna(0.0)
ret_gross = ret_net + cost_frac

# Analytic cost split.
pos = result.positions.fillna(0.0)
trades = pos.diff().fillna(pos.iloc[0])
hb = pd.Series(half_bp)
spread_d = (trades.abs() * hb.reindex(trades.columns) / 1e4).sum(axis=1)
rl, rs = pd.Series(swp_l), pd.Series(swp_s)
swap_d = (-(pos.clip(lower=0) * rl.reindex(pos.columns)
            + pos.clip(upper=0).abs() * rs.reindex(pos.columns)) / 252).sum(axis=1)
print(f"Engine costs ${result.costs.sum():,.0f} | analytic: spread ${spread_d.sum():,.0f} "
      f"+ swap ${swap_d.sum():,.0f} = ${spread_d.sum() + swap_d.sum():,.0f}")
gsh = ret_gross.mean() / ret_gross.std() * np.sqrt(252)
print(f"GROSS Sharpe {gsh:+.2f} (net {ret_net.mean() / ret_net.std() * np.sqrt(252):+.2f}) | "
      f"swap drag {252 * (swap_d / eq.shift(1)).mean():.2%}/yr | "
      f"spread drag {252 * (spread_d / eq.shift(1)).mean():.2%}/yr")

print(f"\n{'year':>5} {'gross':>8} {'gSh':>6} {'net':>8} {'swap$':>10} {'spread$':>9}")
for yr, g in ret_gross.groupby(ret_gross.index.year):
    n = ret_net[ret_net.index.year == yr]
    print(f"{yr:>5} {(1 + g).prod() - 1:>+8.1%} "
          f"{g.mean() / g.std() * np.sqrt(252) if g.std() > 0 else float('nan'):>+6.2f} "
          f"{(1 + n).prod() - 1:>+8.1%} {swap_d[swap_d.index.year == yr].sum():>10,.0f} "
          f"{spread_d[spread_d.index.year == yr].sum():>9,.0f}")

# 2022 autopsy: month-end book vs what trend "should" have held.
KEY = ["EURUSD", "USDJPY", "GBPUSD", "US500", "US100", "GER40", "USOIL", "UKOIL", "XAUUSD"]
w22 = result.weights.loc["2022", KEY]
me = w22.groupby([w22.index.year, w22.index.month]).last()
print("\n2022 month-end weights (sign matters: trend wanted short EUR/GBP, long USDJPY, "
      "short indices, long oil):")
print((me * 100).round(1).to_string())


In [ ]:
# Cell 6 — #014 trial 2: net-of-financing trend (swap inside the signal, zero new params).
# edge_long = r252 + 1y long swap; edge_short = -r252 + 1y short swap.
# Long if edge_long > max(0, edge_short); short if edge_short > max(0, edge_long); else FLAT
# (the trailing trend doesn't cover its own financing). Sizing identical to t1: 0.4/sigma,
# split by data-alive count (not position count), so per-position size matches t1 and the
# book simply drops unprofitable holdings -> lower gross, same per-name risk.
class TSMOMNetCarry(Strategy):
    """Trend signal on net-of-swap trailing return; trade only self-financing sides."""
    rebalance_frequency = staticmethod(_month_end_bars)

    def __init__(self, long_rate: dict, short_rate: dict):
        self.rl = pd.Series(long_rate)
        self.rs = pd.Series(short_rate)
        self.risk = RiskConfig(max_position=0.25, max_gross=6.0)

    def __repr__(self):
        return "TSMOMNetCarry(sign 252d net of per-side swap, 0.40/sigma, monthly)"

    def generate_weights(self, data, t):
        px = data.prices
        if len(px) < LOOKBACK + 1:
            return pd.Series(0.0, index=data.assets)
        r252 = px.iloc[-1] / px.iloc[-(LOOKBACK + 1)] - 1.0
        el = r252 + self.rl.reindex(r252.index)
        es = -r252 + self.rs.reindex(r252.index)
        sig = pd.Series(0.0, index=r252.index)
        sig[(el > 0) & (el >= es)] = 1.0
        sig[(es > 0) & (es > el)] = -1.0
        vol = (px.pct_change(fill_method=None).ewm(span=VOL_SPAN).std().iloc[-1]
               * np.sqrt(252)).clip(lower=VOL_FLOOR)
        n_alive = int(r252.notna().sum())
        if n_alive == 0:
            return pd.Series(0.0, index=data.assets)
        w = (sig * SCALE / vol / n_alive).fillna(0.0)
        return w.reindex(data.assets).fillna(0.0)


strat2 = TSMOMNetCarry(swp_l, swp_s)
print(strat2)
engine2 = Engine(panel, strat2, costs=costs, liquidity=None, initial_capital=1_000_000)
result2 = engine2.run(start=START)

eq2 = result2.equity_curve
ret2 = result2.returns.dropna()
sh2 = ret2.mean() / ret2.std() * np.sqrt(252)
dd2 = (eq2 / eq2.cummax() - 1).min()
pos2 = result2.positions.fillna(0.0)
swap2 = (-(pos2.clip(lower=0) * pd.Series(swp_l).reindex(pos2.columns)
           + pos2.clip(upper=0).abs() * pd.Series(swp_s).reindex(pos2.columns)) / 252).sum(axis=1)
print(f"Final equity ${eq2.iloc[-1]:,.0f} ({eq2.iloc[-1] / 1e6 - 1:+.1%}) | "
      f"net Sharpe {sh2:+.2f} | vol {ret2.std() * np.sqrt(252):.1%} | maxDD {dd2:.1%}")
print(f"Costs ${result2.costs.sum():,.0f} (swap ${swap2.sum():,.0f}) | "
      f"swap drag {252 * (swap2 / eq2.shift(1)).mean():.2%}/yr | "
      f"avg gross {result2.weights.abs().sum(axis=1).mean():.2f} (t1: 4.5)")

cost_frac2 = (result2.costs / eq2.shift(1)).reindex(ret2.index).fillna(0.0)
gross2 = ret2 + cost_frac2
print(f"GROSS Sharpe {gross2.mean() / gross2.std() * np.sqrt(252):+.2f} (t1 gross +0.31)")
print(f"\n{'year':>5} {'net':>8} {'netSh':>6} | {'t1 net':>8}")
t1net = result.returns.dropna()
for yr, g in ret2.groupby(ret2.index.year):
    n1 = t1net[t1net.index.year == yr]
    print(f"{yr:>5} {(1 + g).prod() - 1:>+8.1%} "
          f"{g.mean() / g.std() * np.sqrt(252) if g.std() > 0 else float('nan'):>+6.2f} | "
          f"{(1 + n1).prod() - 1:>+8.1%}")


In [ ]:
# Cell 7 — #014 trial 3 (originally pre-registered): lookback-blend TSMOM.
# signal = sign(mean[sign(63d), sign(126d), sign(252d)]) per instrument; same 0.4/sigma
# sizing, same real FTMO costs. Robustness check on signal speed (does faster trend help?).
LBS = [63, 126, 252]


class TSMOMBlend(Strategy):
    rebalance_frequency = staticmethod(_month_end_bars)

    def __init__(self):
        self.risk = RiskConfig(max_position=0.25, max_gross=6.0)

    def __repr__(self):
        return "TSMOMBlend(mean sign[63,126,252], 0.40/sigma, monthly)"

    def generate_weights(self, data, t):
        px = data.prices
        if len(px) < max(LBS) + 1:
            return pd.Series(0.0, index=data.assets)
        sig = np.sign(sum(np.sign(px.iloc[-1] / px.iloc[-(lb + 1)] - 1.0) for lb in LBS))
        vol = (px.pct_change(fill_method=None).ewm(span=VOL_SPAN).std().iloc[-1]
               * np.sqrt(252)).clip(lower=VOL_FLOOR)
        n_alive = int((px.iloc[-1] / px.iloc[-(max(LBS) + 1)]).notna().sum())
        if n_alive == 0:
            return pd.Series(0.0, index=data.assets)
        return (sig * SCALE / vol / n_alive).fillna(0.0).reindex(data.assets).fillna(0.0)


strat3 = TSMOMBlend()
print(strat3)
engine3 = Engine(panel, strat3, costs=costs, liquidity=None, initial_capital=1_000_000)
result3 = engine3.run(start=START)
eq3 = result3.equity_curve
ret3 = result3.returns.dropna()
cost_frac3 = (result3.costs / eq3.shift(1)).reindex(ret3.index).fillna(0.0)
gross3 = ret3 + cost_frac3
print(f"Final ${eq3.iloc[-1]:,.0f} ({eq3.iloc[-1] / 1e6 - 1:+.1%}) | "
      f"net Sharpe {ret3.mean() / ret3.std() * np.sqrt(252):+.2f} | "
      f"gross Sharpe {gross3.mean() / gross3.std() * np.sqrt(252):+.2f} | "
      f"maxDD {(eq3 / eq3.cummax() - 1).min():.1%} | costs ${result3.costs.sum():,.0f}")

print("\nSession trial summary (net / gross Sharpe):")
print(f"  t1 MOP-exact 252d:      net -0.34 | gross +0.31")
print(f"  t2 net-of-financing:    net -0.17 | gross +0.17")
print(f"  t3 blend[63,126,252]:   net {ret3.mean() / ret3.std() * np.sqrt(252):+.2f} | "
      f"gross {gross3.mean() / gross3.std() * np.sqrt(252):+.2f}")


In [ ]:
# Cell 8 — #014 DSR (best trial) + verdict.
# Best NET trial = t2 (net-of-financing, -0.17). All trials negative net -> DSR on the
# best is the formal nail. K=3 session trials; also report K=8 (incl #011/#012/#013 leads).
from backtest.selection import dsr, expected_max_sr
from backtest.metrics.core import sharpe_var_term

best = result2.returns.dropna()
r = best.to_numpy()
sr_pb = r.mean() / r.std(ddof=1)
vt = sharpe_var_term(r, sr_pb)
print(f"Best trial (t2 net-of-financing): ann net Sharpe {sr_pb * np.sqrt(252):+.2f}")
for K in [1, 3, 8]:
    print(f"  K={K}: luck-max {expected_max_sr(K, vt / len(r)) * np.sqrt(252):+.3f} ann | "
          f"DSR {dsr(best, K=K):.3f}")

print("\n#014 VERDICT: KILLED (all 3 trials net-negative; gross too weak to clear swap)")
print("  t1 MOP-exact:  net -0.34 | gross +0.31 | swap drag 8.3%/yr")
print("  t2 net-carry:  net -0.17 | gross +0.17 | swap drag 3.9%/yr (best net)")
print("  t3 blend:      net -0.43 | gross +0.20 | swap drag ~6%/yr")
print("  Root cause: UNIVERSE, not costs. Gross trend on FTMO-45 ~+0.2-0.3 Sharpe and a")
print("  NEGATIVE-gross decade 2015-23; the published TSMOM profit lives in bonds/rates +")
print("  commodity breadth that FTMO does not list. Need gross ~0.6 to clear swap+matter.")
print("  Implementation validated (2022 book correctly short EUR/GBP, long JPY, short indices).")
